# 05 — CLI Quick Reference

This notebook documents the `evalf` CLI commands and options.
Each cell contains the command you would run in a terminal.

## Prerequisites

```bash
# Install from PyPI
pip install evalf

# Or from source
git clone https://github.com/hnhoangdz/evalf.git
cd evalf
uv sync --extra dev
```

Configure your judge model:

```bash
cp .env.example .env.local
# Edit .env.local with your API key and preferred model
```

## List available metrics

```bash
evalf list-metrics
```

Output:
```
answer_correctness
answer_relevance
c4
context_coverage
context_precision
context_recall
context_relevance
faithfulness
```

## Evaluate a single sample

```bash
evalf run \
  --question "Under FERPA, when do rights transfer from parents to a student?" \
  --retrieved-context "When a student turns 18 years old, or enters a postsecondary institution at any age, the rights under FERPA transfer from the parents to the student." \
  --actual-output "Under FERPA, rights transfer when a student turns 18 or enters a postsecondary institution at any age." \
  --expected-output "Under FERPA, rights transfer when a student turns 18 years old or enters a postsecondary institution at any age." \
  --metrics faithfulness,answer_correctness,answer_relevance \
  --threshold 0.7
```

Use `--retrieved-context` multiple times for multiple contexts:

```bash
evalf run \
  --question "..." \
  --retrieved-context "First context" \
  --retrieved-context "Second context" \
  --actual-output "..." \
  --metrics faithfulness
```

## Evaluate inline JSON

```bash
evalf run \
  --sample-json '{"id":"case-1","question":"What does FERPA protect?","retrieved_contexts":["FERPA protects the privacy of student education records."],"actual_output":"FERPA protects student records.","expected_output":"FERPA protects the privacy of student education records."}'
```

## Evaluate a dataset file

Supports `.jsonl` and `.json` files.

```bash
evalf run \
  --input examples/rag_eval.jsonl \
  --metrics faithfulness,answer_correctness \
  --threshold 0.8 \
  --output report.json
```

Markdown reports:

```bash
evalf run \
  --input examples/rag_eval.jsonl \
  --metrics answer_correctness \
  --output report.md
```

## Use the C4 composite metric

```bash
evalf run \
  --input examples/rag_eval.jsonl \
  --metrics c4 \
  --threshold 0.7
```

C4-specific flags:

```bash
# Request a second judge call to synthesize one summary reason
evalf run --input data.jsonl --metrics c4 --c4-summary-reason

# Omit per-criterion reasoning from the report
evalf run --input data.jsonl --metrics c4 --no-c4-include-reason

# Strict mode: clamp below-threshold scores to 0.0
evalf run --input data.jsonl --metrics c4 --c4-strict-mode
```

## Multi-attempt evaluation

`pass@k` — the sample passes if **any** attempt passes.
`pass^k` — the sample passes only if **all** evaluated attempts pass.

```bash
evalf run \
  --input examples/rag_eval_attempts.json \
  --metrics faithfulness,answer_correctness \
  --metric-mode pass@k \
  --k 3
```

```bash
evalf run \
  --input examples/rag_eval_attempts.json \
  --metrics answer_correctness \
  --metric-mode pass^k \
  --k 2
```

## Per-metric threshold overrides

Set a global threshold with `--threshold` and override individual metrics:

```bash
evalf run \
  --input examples/rag_eval.jsonl \
  --metrics faithfulness,answer_correctness,answer_relevance \
  --threshold 0.7 \
  --metric-threshold faithfulness=0.9 \
  --metric-threshold answer_correctness=0.8
```

## Provider and model overrides

```bash
# Use a different model
evalf run --input data.jsonl --provider openai --model gpt-4.1

# Use Gemini
evalf run --input data.jsonl --provider gemini --model gemini-2.5-flash

# Use Claude via an OpenAI-compatible proxy
evalf run --input data.jsonl --provider claude --model claude-sonnet-4 --base-url https://your-proxy.com/v1
```

## Timeout and concurrency

```bash
evalf run \
  --input examples/rag_eval_mixed.jsonl \
  --metrics faithfulness,answer_correctness \
  --concurrency 8 \
  --request-timeout-seconds 30 \
  --per-sample-timeout-seconds 120 \
  --temperature 0.0 \
  --max-tokens 600
```

## Environment variables

All CLI options have environment-variable equivalents:

| Variable | Description |
|----------|-------------|
| `EVALF_PROVIDER` | Judge provider (openai, gemini, claude) |
| `EVALF_MODEL` | Judge model name |
| `EVALF_BASE_URL` | OpenAI-compatible base URL |
| `EVALF_API_KEY` | API key (also reads `OPENAI_API_KEY`, `GEMINI_API_KEY`, `ANTHROPIC_API_KEY`) |
| `EVALF_CONCURRENCY` | Max concurrent samples |
| `EVALF_REQUEST_TIMEOUT_SECONDS` | Per-request timeout |
| `EVALF_PER_SAMPLE_TIMEOUT_SECONDS` | Per-sample timeout |
| `EVALF_MAX_RETRIES` | Retry count (0–10) |
| `EVALF_TEMPERATURE` | Sampling temperature |
| `EVALF_MAX_TOKENS` | Max completion tokens |
| `EVALF_THRESHOLD` | Default pass/fail threshold |
| `EVALF_METRICS` | Comma-separated default metrics |
| `EVALF_METRIC_MODE` | `pass@k` or `pass^k` |
| `EVALF_K` | Number of attempts (1–5) |
| `EVALF_OUTPUT` | Default output path |

`evalf` reads `.env.local` first, then `.env`. CLI flags always take priority.